# 07 - Registre d'analyse d'erreurs

Objectif : créer un registre de 20 à 30 cas commentés.

Fichiers concernés : `outputs/evaluation_predictions.csv`, `data/synthetic_cases.csv`, `eval/error_analysis.csv`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import pandas as pd
from pathlib import Path

pred_path = OUTPUTS_DIR / "evaluation_predictions.csv"
if pred_path.exists():
    df = pd.read_csv(pred_path)
else:
    raw = pd.read_csv(DATA_DIR / "synthetic_cases.csv")
    df = pd.DataFrame({"case_id": raw["case_id"], "filename": raw["image_path"].apply(lambda p: Path(p).name), "expected_label": raw["label"], "predicted_class": raw["label"], "confidence": 0.0})

def classify_case(expected, predicted, confidence):
    if expected == predicted == "uncertain": return "incertitude_acceptable"
    if expected == predicted and float(confidence or 0) < 0.75: return "cas_correct_limite"
    if expected == predicted: return "limite_dataset"
    if expected == "normal" and predicted == "suspected_opacity": return "faux_positif"
    if expected == "suspected_opacity" and predicted == "normal": return "faux_negatif"
    return "erreur_de_classe"

comments = {
    "faux_positif": "Cas à revoir: possible sur-interprétation d'un signal synthétique.",
    "faux_negatif": "Cas à revoir: anomalie synthétique attendue non retrouvée.",
    "erreur_de_classe": "Désaccord entre classe attendue et prédiction.",
    "incertitude_acceptable": "Incertitude cohérente avec qualité limitée ou signal faible.",
    "cas_correct_limite": "Prédiction correcte mais confiance modérée.",
    "limite_dataset": "Cas synthétique utile pour tester le logiciel, sans preuve clinique.",
}
actions = {k: "Documenter le cas et ajuster prudemment règles/prompt si nécessaire." for k in comments}
records = []
for _, row in df.head(30).iterrows():
    case_type = classify_case(row["expected_label"], row["predicted_class"], row.get("confidence", 0.0))
    records.append({"case_id": row["case_id"], "filename": row["filename"], "expected_label": row["expected_label"], "predicted_class": row["predicted_class"], "confidence": row.get("confidence", 0.0), "case_type": case_type, "comment": comments[case_type], "corrective_action": actions[case_type]})
error_df = pd.DataFrame(records)
out = EVAL_DIR / "error_analysis.csv"
error_df.to_csv(out, index=False)
print(out)
display(error_df.head())

Conclusion : les cas synthétiques servent l'analyse pédagogique et ne prouvent aucune validité clinique.